#### Setup

This is the **set-up cell** - run it first.

* Qiskit has changed a lot over the years, and some older commands (`Aer`, `BasicAer`, `execute`,
  `IBMQ`) no longer exist. The helper file `qiskit_backend_config.py` (in this folder) provides
  stand-ins for them, so the older code below still works.
* `sys.path.insert(...)` tells Python to look for that helper file in the current folder.
* `BACKEND_NAME = "qasm_simulator"` chooses **where the circuits run**. A *backend* is either a
  simulator on your own computer or a real quantum computer at IBM. The `qasm_simulator` imitates
  a real device: it *measures* the qubits many times ("shots") and counts the results.
* If you have an IBM Quantum account you could set a token and pick a real device, but nothing in
  this notebook needs that.
* The last line prints which backend was created, so you know the cell worked.

In [2]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')

Backend ready: LegacyAerSimulator('aer_simulator')


This cell tries to **log in to IBM Quantum** with `IBMQ.load_account()`.

* `IBMQ` here is the stand-in from `qiskit_backend_config.py`. If no IBM token is set up on your
  computer, it simply prints a warning and keeps using the local simulator, so it is safe to run.
* Nothing later in this notebook needs a real quantum computer.

In [3]:
#Custom gates
#Amitava Chakraborty - 18th Jan 2018
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

Next cell **imports** (loads) the tools in the notebook.

* `matplotlib` draws plots; `%matplotlib inline` shows them inside the notebook.
* `numpy` (as `np`) does fast maths on arrays of numbers; `pi` is $\pi \approx 3.14159$.
* From Qiskit:
  * `QuantumCircuit`, `QuantumRegister`, `ClassicalRegister` build circuits;
  * `Parameter` is a placeholder for an angle whose value we choose later (like a variable $x$
    in algebra);
  * `generate_preset_pass_manager` prepares circuits for running (see below);
  * `SparsePauliOp` describes the *quantity we want to measure* (an "observable");
  * `StatevectorEstimator` is the **Estimator primitive**, the star of this notebook.
the Estimator does its own simulation.

In [4]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from math import pi
# Importing standard Qiskit libraries and configuring account
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray
from qiskit.primitives import StatevectorEstimator

Next cell **builds a 2-qubit circuit with two adjustable angles**.

* `QuantumCircuit(2)` creates a circuit with qubits 0 and 1, both starting in $|0\rangle$.
* `h(0)` puts qubit 0 into a superposition; `cx(0, 1)` (CNOT) then **entangles** the two qubits.
* `ry(Parameter("a"), 0)` and `rz(Parameter("b"), 0)` rotate qubit 0 by angles **a** and **b**
  around the Y and Z axes of the Bloch sphere. The angles have no value yet; they are *parameters*.
* `cx(0, 1)` and `h(0)` undo the entangling step, so how the final state looks depends on a and b.

Why parameters? It lets us run *the same circuit* for many different angles without rebuilding it.
Parameterised circuits are the heart of variational algorithms and quantum machine learning.

In [5]:
# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

Next code cell **transpiles** the circuit.

* Real quantum hardware understands only a few basic gates and only some qubit connections.
  *Transpiling* rewrites the circuit into an equivalent one that the target can run, and tidies it
  up (`optimization_level=1` means "light optimisation").
* `generate_preset_pass_manager` creates the transpiler; `pm.run(circuit)` applies it.
* `layout` records which *physical* qubit each of our *logical* qubits ended up on. We need it in
  the next cell, so that the observables refer to the right qubits.

In [6]:
# Transpile the circuit without providing a backend
pm = generate_preset_pass_manager(optimization_level=1)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

Next code cell prepares **the inputs for the Estimator**: the angle values and the things to measure.

**1. The angles (`params`).**
`np.linspace(-np.pi, np.pi, 10)` makes 10 evenly spaced values of **a** from $-\pi$ to $\pi$;
the second `linspace` makes 10 values of **b** from $-4\pi$ to $4\pi$. `np.vstack(...).T` puts
them side by side, giving a table with **10 rows × 2 columns**: each row is one pair (a, b).

**2. The observables.**
An *observable* is a quantity we can measure. Here they are written with **Pauli letters**, one
letter per qubit:

* `"XX"`: measure $X$ on both qubits and multiply the results;
* `"IY"`: $I$ (do nothing) on one qubit, $Y$ on the other;
* `SparsePauliOp(["XX", "IY"], [0.5, 0.5])` is the combination $0.5\,XX + 0.5\,IY$.

The extra square brackets give the list the shape (3, 1): 3 observables in a column. Combined
with the 10 parameter rows, the Estimator will compute every observable for every parameter row.

**3. `apply_layout(layout)`** moves each observable onto the same physical qubits that the
transpiler used in the previous cell.

In [7]:
# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 10),
        np.linspace(-4 * np.pi, 4 * np.pi, 10),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

Next code cell **creates the Estimator and packs the job**.

* The **Estimator** is one of Qiskit's two *primitives* (the other is the Sampler). You give it a
  circuit and an observable, and it returns the **expectation value**: the *average* result you
  would get if you prepared the state and measured the observable many times.
* `StatevectorEstimator()` computes this exactly with a simulation of the state (no measurement
  noise).
* A **PUB** ("primitive unified bloc") bundles the three inputs: `(circuit, observables, parameter values)`.

> **Note:** the comment in the cell says "300 combinations" and shape "(3, 100)". With the 10
> parameter rows created above it is actually **3 × 10 = 30** combinations and shape **(3, 10)**,
> as the printed results below confirm.

In [8]:
# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator = StatevectorEstimator()
estimator_pub = (transpiled_circuit, observables, params)

Next code cell **runs the Estimator**.

* `estimator.run([estimator_pub])` submits the job. The square brackets mean you could submit
  several PUBs at once; here there is one.
* `job.result()` waits for the job to finish and returns the results.

In [9]:
# Run the transpiled circuit
# using the set of parameters and observables.

job = estimator.run([estimator_pub])
result = job.result()

**prints an overview of the result**.

* `len(result)` is the number of PUBs we submitted (1).
* The printed text shows the result's structure: a `PubResult` holding a `DataBin` with arrays
  `evs` and `stds` of shape (3, 10): 3 observables × 10 parameter pairs.
* The `f"..."` is a Python *f-string*: anything inside `{ }` is replaced by its value.

In [10]:
print(
    f"The result of the submitted job had {len(result)} PUB and "
    f"has a value:\n {result}\n"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10)), metadata={'target_precision': 0.0, 'circuit_metadata': {}})], metadata={'version': 2})



Let us now **look inside the result of our single PUB**.

* `result[0]` is the first (and only) PUB result; `.data` is its **DataBin**, a container holding
  the computed numbers.

In [11]:
print(
    f"The associated PubResult of this job has the following data bins:"
    f"\n {result[0].data}\n"
)

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 10), dtype=float64>), stds=np.ndarray(<shape=(3, 10), dtype=float64>), shape=(3, 10))



**list what the DataBin contains**:

* `evs`: the **e**xpectation **v**alue**s**;
* `stds`: their **st**andard **d**eviations (the uncertainty). The exact `StatevectorEstimator`
  has no measurement noise, so these are essentially 0.

In [12]:
print(f"And this DataBin has attributes: {result[0].data.keys()}")

And this DataBin has attributes: dict_keys(['evs', 'stds'])


**prints a reminder** of where the shape of the result comes from.

> The printed sentence says "(100, 2)", but our parameter table has shape **(10, 2)**: 10 pairs
> of (a, b). Combined with the 3 observables, that gives results of shape **(3, 10)**.

In [13]:
print(
    "Recall that this shape is due to our array of parameter binding sets "
    "having shape (100, 2) -- where 2 is the number of parameters in the circuit -- "
    "combined with our array of observables having shape (3, 1)."
)

Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the number of parameters in the circuit -- combined with our array of observables having shape (3, 1).


Now let us see **the expectation values themselves**: a table of 3 rows × 10 columns.

* **Row** = which observable ($0.5\,XX + 0.5\,IY$, then $XX$, then $IY$).
* **Column** = which (a, b) pair.
* Every value lies between $-1$ and $+1$ (Pauli measurements give $\pm 1$, and the expectation value
  is their average). Numbers like `3e-16` are just tiny rounding errors, i.e. **0**.
* Check: row 1 should equal half of row 2 plus half of row 3. You can verify this with a few columns.

In [14]:

print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The expectation values measured from this PUB are: 
[[ 3.06161700e-16  4.52395120e-01  4.36594428e-01  2.16506351e-01
   6.33718361e-01 -6.33718361e-01 -2.16506351e-01 -4.36594428e-01
  -4.52395120e-01 -3.06161700e-16]
 [ 1.22464680e-16  6.42787610e-01  9.84807753e-01  8.66025404e-01
   3.42020143e-01 -3.42020143e-01 -8.66025404e-01 -9.84807753e-01
  -6.42787610e-01 -1.22464680e-16]
 [ 4.89858720e-16  2.62002630e-01 -1.11618897e-01 -4.33012702e-01
   9.25416578e-01 -9.25416578e-01  4.33012702e-01  1.11618897e-01
  -2.62002630e-01 -4.89858720e-16]]
